# S5.G2-1 · G3 臂 A_secure：隐语 SS-XGB 训练记录、续训核对与打分

> 按冻结的 S6.P3：A_secure 沿用 A 的配置（copy_of_A，不调参），只在 Santander，单次拟合上限 6 小时。运行方式与分段续训见 DR-M5-010：两个本机进程各代表一方（隐语生产模式），每段由 `g3_ssxgb.run_chunk` 启动、逐棵存档，续训时回放已训练的树。
> 训练在后台分 6 次启动完成（会话后台任务单次上限 2 小时）；本 notebook 汇总训练记录、重跑续训核对并对开发评估集打分。首次启动（10/28 20:58）在第 4 棵后中止、检查点弃用、未打分（DR-M5-010 context ⑤），不在下表内。

In [1]:
import hashlib
import json
import os
import pathlib
import subprocess
import sys

import numpy as np
import pandas as pd
import yaml

HASH_PREFIX_LEN = 12
BYTES_PER_GIB = 1024 ** 3
D4_MIN_ABS_DIFF = 0.005          # DR-M5-010 · D-4：「可分辨」但差值小于此值时降级

REPO_ROOT = pathlib.Path.cwd()
while not (REPO_ROOT / "AGENTS.md").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

CONFIG_PATH = REPO_ROOT / "modules/m5_modeling/configs/s5_g2_ssxgb_run.yaml"
run = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
PRE_PATH = REPO_ROOT / run["preregistration"]
pre = yaml.safe_load(PRE_PATH.read_text(encoding="utf-8"))
fz = pre["freeze"]
ad = fz["arm_details"]
SSXGB_PATH = REPO_ROOT / "modules/m5_modeling/components/g3_ssxgb.py"


def sha(path, n=HASH_PREFIX_LEN):
    return hashlib.sha256(pathlib.Path(path).read_bytes()).hexdigest()[:n]


git_sha = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True, cwd=REPO_ROOT).strip()
dirty = subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True,
                       cwd=REPO_ROOT).stdout.strip() != ""
print("config:", CONFIG_PATH.relative_to(REPO_ROOT), "sha256:" + sha(CONFIG_PATH))
print("preregistration:", PRE_PATH.relative_to(REPO_ROOT), "sha256:" + sha(PRE_PATH), "| status:", pre["status"])
print("A_secure:", pre["arms"]["A_secure"], "| 数据集:", pre["compute_caps"]["a_secure_datasets"])
print("seed:", run["seed"], "| seeds:", run["seeds"])
print("git:", git_sha + ("（工作区含本步未提交改动）" if dirty else ""))
print("step:", run["step_id"], "|", run["purpose"])
print("SS-XGB 组件:", SSXGB_PATH.relative_to(REPO_ROOT), "sha256:" + sha(SSXGB_PATH))
mismatch = [p for p, h in fz["code_sha256"].items() if hashlib.sha256((REPO_ROOT / p).read_bytes()).hexdigest() != h]
print("冻结代码指纹核对：", "一致" if not mismatch else "不一致 " + str(mismatch))
assert pre["status"] == "frozen" and not mismatch
PRIV = pathlib.Path(os.environ[run["private_root_env"]])
SEED0 = run["seeds"][0]
MB, NJ = ad["xgboost_max_bin"], ad["n_jobs"]

config: modules/m5_modeling/configs/s5_g2_ssxgb_run.yaml sha256:0ad22448d9fc
preregistration: modules/m6_evaluation/configs/s6_p3_g3_preregistration.yaml sha256:c0a4bbfc9709 | status: frozen
A_secure: {'who_trains': 'both', 'model': 'secretflow_ss_xgb', 'flow': 'secret_shared', 'config': 'copy_of_A'} | 数据集: ['santander']
seed: 11 | seeds: [11, 22, 33, 44, 55, 66]
git: f634ff3（工作区含本步未提交改动）
step: S5.G2 | 按冻结的 S6.P3 在 Santander 上训练 A_secure（SS-XGB，全程秘密分享），与 A（SGB）做配对比较并报告成本
SS-XGB 组件: modules/m5_modeling/components/g3_ssxgb.py sha256:0f55ee4a8826
冻结代码指纹核对： 一致


In [2]:
from modules.m5_modeling.components import g3_sgb as SG
from modules.m5_modeling.components import g3_ssxgb as SX

TUNING = json.loads((PRIV / run["tuning_file"]).read_text(encoding="utf-8"))
params = SG.sgb_params(TUNING["A"]["best"], ad["sgb_sketch_eps"], run["seed"])
TOTAL, CAP = params["num_boost_round"], pre["compute_caps"]["max_wall_clock_hours_per_fit"]
CKPT = PRIV / run["ckpt_dir"]
print("A_secure 参数（与 A 相同，copy_of_A，不调参）：", params)
chunks = [json.loads(line) for line in (CKPT / "chunks.jsonl").read_text(encoding="utf-8").splitlines()]
assert all(c["component_sha256"] == hashlib.sha256(SSXGB_PATH.read_bytes()).hexdigest() for c in chunks)
assert all(c["params"] == params for c in chunks)
results = [c for c in chunks if c["event"] == "result"]
assert len(results) == sum(c["event"] == "launch" for c in chunks)      # 每次启动都有结果记录，没有中途丢失的段
cols = ["started", "resume_from", "stop_at", "trees_done", "cap_reached", "wall_seconds", "exit_codes"]
print("训练分段（每段由 g3_ssxgb.run_chunk 启动；组件指纹与参数逐段核对一致）：")
print(pd.DataFrame([{k: c[k] for k in cols} for c in results]).round(1).to_string(index=False))
prog = {p: SX.load_progress(str(CKPT), p) for p in SX.PARTIES}
assert prog["neidi"]["trees_done"] == prog["hk"]["trees_done"]
COST = SX.cost_summary(prog[SX.LABEL_PARTY], TOTAL, CAP)
T = COST["trees"]
tree_s = np.asarray(prog[SX.LABEL_PARTY]["tree_seconds"])
seg_mean = [float(tree_s[c["start"]:c["end"]].mean()) for c in prog[SX.LABEL_PARTY]["chunks"]]
peak = max(SX.cost_summary(prog[p], TOTAL, CAP)["peak_rss_mb"] for p in SX.PARTIES)
print("已训 %d / %d 棵；拟合墙钟 %.0f 秒（%.2f 小时，上限 %s 小时）；完成且未超限：%s"
      % (T, TOTAL, COST["fit_seconds"], COST["fit_seconds"] / SX.SECONDS_PER_HOUR, CAP, COST["within_cap"]))
print("每棵平均 %.1f 秒；按均值外推 %d 棵约 %.2f 小时；各段平均每棵（秒）：%s"
      % (COST["mean_tree_seconds"], TOTAL, COST["mean_tree_seconds"] * TOTAL / SX.SECONDS_PER_HOUR,
         [round(s, 1) for s in seg_mean]))
print("续训开销（重做准备与回放）%.0f 秒；本机回环发送合计 %.0f GiB，每棵约 %.1f GiB；两方进程峰值内存 %.0f MB"
      % (COST["resume_overhead_seconds"], COST["lo_tx_bytes"] / BYTES_PER_GIB, COST["lo_tx_bytes"] / T / BYTES_PER_GIB,
         peak))

A_secure 参数（与 A 相同，copy_of_A，不调参）： {'num_boost_round': 150, 'max_depth': 5, 'learning_rate': 0.1, 'reg_lambda': 10.0, 'sketch_eps': 0.03, 'objective': 'logistic', 'seed': 11}
训练分段（每段由 g3_ssxgb.run_chunk 启动；组件指纹与参数逐段核对一致）：
                 started  resume_from  stop_at  trees_done  cap_reached  wall_seconds            exit_codes
2026-10-28T21:13:00+0800            0       40          40        False        6490.2 {'hk': 0, 'neidi': 0}
2026-10-28T23:01:24+0800           40       79          79        False        6049.5 {'hk': 0, 'neidi': 0}
2026-10-29T00:42:29+0800           79      119         119        False        6173.6 {'hk': 0, 'neidi': 0}
2026-10-29T02:25:39+0800          119      139         139        False        3092.4 {'hk': 0, 'neidi': 0}
2026-10-29T03:17:25+0800          139      141         141        False         582.2 {'hk': 0, 'neidi': 0}
2026-10-29T03:27:26+0800          141      141         141         True           0.0                    {}
已训 141 / 150 棵；拟合墙钟 21

**读法**：训满 150 棵按均值需约 6.35 小时（每棵约 152 秒），超过预注册单次拟合上限 6 小时。按 DR-M5-010 · D-3，启动方不开始会越过上限的树：训到第 141 棵（拟合墙钟 5.97 小时）后停止，A_secure 记为**超出单次拟合上限、未完成**。各段平均每棵 161.9 → 140.2 秒，后续段没有变慢，D-3 推翻条件未触发。另两项成本：两方每棵树经本机回环合计发送约 32 GiB（141 棵约 4.4 TiB）；两方进程峰值内存约 3.3 GB。续训开销（重做准备与回放）880 秒，单列，不计入拟合墙钟。

In [3]:
import shutil

rc = run["resume_check"]
RC = PRIV / run["resume_check_dir"]
shutil.rmtree(RC, ignore_errors=True)
base = {"python": sys.executable, "repo": str(REPO_ROOT), "arrays": str(PRIV / run["arrays_file"]), "prefix": "train",
        "params": dict(params, num_boost_round=rc["trees"]), "fed_ports": run["fed_ports"],
        "spu_ports": run["spu_ports"], "protocol": run["spu_protocol"], "field": run["spu_field"], "n_rows": rc["n_rows"]}


def cfg(name, **kw):
    return dict(base, ckpt=str(RC / name), **kw)


codes = [SX.launch("train", cfg(n), 0, rc["trees"]) for n in ("u1", "u2")]
for name, replay in (("r", True), ("nr", False)):
    codes.append(SX.launch("train", cfg(name, replay=replay), 0, rc["resume_at"]))
    codes.append(SX.launch("train", cfg(name, replay=replay), rc["resume_at"], rc["trees"]))
P = {}
for name in ("u1", "u2", "r", "nr"):     # 打分用内层验证集前 n_rows 行，不接触标签
    out = RC / name / "pred.npy"
    codes.append(SX.launch("predict", cfg(name, prefix="inner_val"), rc["trees"], rc["trees"], out=str(out)))
    P[name] = np.load(out)
assert all(v == 0 for c in codes for v in c.values())
roots = pd.DataFrame({n: SX.root_splits(str(RC / n), rc["trees"]) for n in P})
same = pd.DataFrame({"u2 与 u1": roots["u2"] == roots["u1"], "r 与 u1": roots["r"] == roots["u1"],
                     "nr 与 u1": roots["nr"] == roots["u1"], "nr 与本组第 1 棵": roots["nr"] == roots["nr"][0]})
same.index.name = "树序号"
print("各棵树根节点（持有方、特征、分裂值）是否相同——u1/u2 不间断，r 在第 %d 棵后续训，nr 不回放对照；"
      "分裂值属模型参数，只比较不打印" % rc["resume_at"])
print(same.to_string())
tol = run["prediction_diff_tolerance"]
share = {n: float((np.abs(P[n] - P["u1"]) > tol).mean()) for n in P}
print("与 u1 预测差超过 %s 的行占比：" % tol, {n: round(v, run["round_auc"]) for n, v in share.items()})
print("平均预测：", {n: round(float(p.mean()), run["round_auc"]) for n, p in P.items()})
CHECK = {"续训根节点逐棵相同": bool((roots["r"] == roots["u1"]).all()),
         "续训差异不超过两次不间断之间的 2 倍": share["r"] <= 2 * share["u2"],
         "不回放对照重复第 1 棵根节点": roots["nr"][rc["resume_at"]] == roots["nr"][0]}
print(CHECK)
assert all(CHECK.values())

[2026-10-29 04:07:00.522] [info] [bigint_spi.cc:79] The default library used for BigInt operations is openssl


各棵树根节点（持有方、特征、分裂值）是否相同——u1/u2 不间断，r 在第 4 棵后续训，nr 不回放对照；分裂值属模型参数，只比较不打印
     u2 与 u1  r 与 u1  nr 与 u1  nr 与本组第 1 棵
树序号                                       
0       True    True     True         True
1       True    True     True        False
2       True    True     True        False
3       True    True     True        False
4       True    True    False         True
5       True    True    False        False
6       True    True     True        False
7       True    True    False        False
与 u1 预测差超过 0.001 的行占比： {'u1': 0.0, 'u2': 0.0174, 'r': 0.0168, 'nr': 1.0}
平均预测： {'u1': 0.1584, 'u2': 0.1584, 'r': 0.1586, 'nr': 0.1275}
{'续训根节点逐棵相同': True, '续训差异不超过两次不间断之间的 2 倍': True, '不回放对照重复第 1 棵根节点': True}


**读法**：DR-M5-010 · D-2 的三条核对都成立。①续训的 8 棵树根节点与不间断训练逐棵相同；②续训与不间断训练预测差超过 0.001 的行占 1.7%，两次不间断训练之间同为 1.7%，未超过 2 倍；③不回放对照（隐语原生续训）在序号 4 的树上重复第 1 棵的根节点，此后根节点与不间断训练分叉，全部行预测都不同，平均预测偏低（0.128 对 0.158）。两次不间断训练本身就不完全相同（D-4）：秘密分享定点运算的舍入会让接近并列的分裂翻转。核对只用训练集与内层验证集前 5,000 行，不接触开发评估集。

In [4]:
SCORES = PRIV / run["ssxgb_scores_file"]
full = dict(base, params=params, n_rows=None, ckpt=str(CKPT), prefix="dev_eval")
codes = SX.launch("predict", full, T, T, out=str(SCORES))
assert all(v == 0 for v in codes.values())
pred = np.load(SCORES)
arr = np.load(PRIV / run["arrays_file"])
assert pred.shape == arr["dev_eval__y"].shape
last = SX.load_progress(str(CKPT), SX.LABEL_PARTY)["predict"][-1]
print("A_secure（%d 棵）对开发评估集 %d 位客户打分，分数只揭示给内地；耗时 %.0f 秒" % (last["trees"], len(pred), last["seconds"]))

A_secure（141 棵）对开发评估集 20351 位客户打分，分数只揭示给内地；耗时 56 秒


**读法**：141 棵模型对开发评估集 20,351 位客户打分用时 56 秒，分数只揭示给内地；逐行分数只存私有目录，不入仓库。